![electronic_medical_records](electronic_medical_records.png)

Medical professionals often summarize patient encounters in transcripts written in natural language, which include details about symptoms, diagnosis, and treatments. These transcripts can be used for other medical documentation, such as for insurance purposes, but as they are densely packed with medical information, extracting the key data accurately can be challenging.  

You and your team at Lakeside Healthcare Network have decided to leverage the OpenAI API to automatically extract medical information from these transcripts and automate the matching with the appropriate ICD-10 codes. ICD-10 codes are a standardized system used worldwide for diagnosing and billing purposes, such as insurance claims processing.

## The Data
The dataset contains anonymized medical transcriptions categorized by specialty.

## transcriptions.csv
| Column     | Description              |
|------------|--------------------------|
| `"medical_specialty"` | The medical specialty associated with each transcription.  |
| `"transcription"` | Detailed medical transcription texts, with insights into the medical case. |

In [ ]:
# Import the necessary libraries
import pandas as pd
from openai import OpenAI
import json

: 

In [ ]:
# Load the data
df = pd.read_csv("data/transcriptions.csv")
df.head()

,medical_specialty,transcription
0,Allergy / Immunology,"SUBJECTIVE:, This 23-year-old white female pr..."
1,Orthopedic,"CHIEF COMPLAINT:, Achilles ruptured tendon.,H..."
2,Bariatrics,"PREOPERATIVE DIAGNOSIS: , Morbid obesity.,POST..."
3,Cardiovascular / Pulmonary,"PREOPERATIVE DIAGNOSES,Airway obstruction seco..."
4,Urology,"CHIEF COMPLAINT:, Urinary retention.,HISTORY ..."


In [ ]:
## Start coding here, use as many cells as you need
# Initialize the OpenAI client: make sure you have a valid API key named OPENAI_API_KEY in your Environment Variables
# Assuming OpenAI API key is set in the environment
import os
api_key = os.environ['OPENAI_API_KEY']
# print('openai_api_key', api_key)
client = OpenAI(api_key=api_key)

: 

Define a list of function definitions that used in tool calls of OpenAI. Namely, create a function definition with the function name='extract_age_specialty' and description='Extract age and medical specialty from the transcription'. That function include two parameters: 'age' is a string for 'Patient name', 'medical specialty' is a string for 'Medical specialty that patient is treating'

In [ ]:
# Define the function for OpenAI tool calls with 'recommended_treatment' instead of 'medical_specialty'
function_definitions = [
    {
        "name": "extract_age_specialty",
        "description": "Extract age and recommended treatment from the transcription",
        "parameters": {
            "type": "object",
            "properties": {
                "age": {
                    "type": "string",
                    "description": "Patient age"
                },
                "recommended_treatment": {
                    "type": "string",
                    "description": "Recommended treatment for the patient"
                }
            }
        }
    }
]

tool_calls = [ {'type': 'function', 'function': func_def} for func_def in function_definitions ]

print("Tool calls defined:", tool_calls)

Tool calls defined: [{'type': 'function', 'function': {'name': 'extract_age_specialty', 'description': 'Extract age and recommended treatment from the transcription', 'parameters': {'type': 'object', 'properties': {'age': {'type': 'string', 'description': 'Patient age'}, 'recommended_treatment': {'type': 'string', 'description': 'Recommended treatment for the patient'}}}}}]


Get the first sample transcription from the 'medical_transcription' column from `df`. Then, create a chat completion request to use the function definition and print the arguments of function call

In [ ]:
# Get the first sample transcription from the 'transcription' column
sample_transcription = df['transcription'].iloc[0]

# Print the sample transcription
print("Sample Transcription:", sample_transcription)

Sample Transcription: SUBJECTIVE:,  This 23-year-old white female presents with complaint of allergies.  She used to have allergies when she lived in Seattle but she thinks they are worse here.  In the past, she has tried Claritin, and Zyrtec.  Both worked for short time but then seemed to lose effectiveness.  She has used Allegra also.  She used that last summer and she began using it again two weeks ago.  It does not appear to be working very well.  She has used over-the-counter sprays but no prescription nasal sprays.  She does have asthma but doest not require daily medication for this and does not think it is flaring up.,MEDICATIONS: , Her only medication currently is Ortho Tri-Cyclen and the Allegra.,ALLERGIES: , She has no known medicine allergies.,OBJECTIVE:,Vitals:  Weight was 130 pounds and blood pressure 124/78.,HEENT:  Her throat was mildly erythematous without exudate.  Nasal mucosa was erythematous and swollen.  Only clear drainage was seen.  TMs were clear.,Neck:  Supple

In [ ]:
messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": sample_transcription}
    ]

In [ ]:
import json

def extract_age_and_recommended_treatment(messages, client, function_definitions):
  """
  Uses OpenAI function calling to extract age and recommended treatment from a transcription.
  Returns a tuple: (age, recommended_treatment).
  """
  response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages,
    functions=function_definitions,
  )
  arguments = response.choices[0].message.function_call.arguments
  parsed_args = json.loads(arguments)
  age = parsed_args.get('age')
  recommended_treatment = parsed_args.get('recommended_treatment')
  return age, recommended_treatment

# Example usage:
age, recommended_treatment = extract_age_and_recommended_treatment(messages, client, function_definitions)
print("Age:", age)
print("Recommended Treatment:", recommended_treatment)

Age: 23
Recommended Treatment: Try Zyrtec instead of Allegra; use loratadine; Samples of Nasonex two sprays in each nostril.


In [ ]:
def get_icd10_codes_for_treatment(recommended_treatment, client):
    """
    Given a recommended treatment, query OpenAI to get the most relevant ICD-10 codes.
    """
    icd_query = f"What are the most relevant ICD-10 codes for the recommended treatment: {recommended_treatment}?"
    icd_messages = [
        {"role": "system", "content": "You are a medical coding assistant. Given a recommended treatment, return the answer as a list of codes like J30.9, R05. Please only include the codes and no other information."},
        {"role": "user", "content": icd_query}
    ]
    icd_response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=icd_messages
    )
    return icd_response.choices[0].message.content

# Example usage:
print("ICD-10 Codes for", recommended_treatment + ":\n", get_icd10_codes_for_treatment(recommended_treatment, client))

ICD-10 Codes for Try Zyrtec instead of Allegra; use loratadine; Samples of Nasonex two sprays in each nostril.:
 J30.9, R09.81, J04.9


In [ ]:
# Process all transcriptions to extract age, medical specialty, recommended treatment, and ICD-10 codes
structured_data = []

for idx, row in df.iterrows():
    transcription = row['transcription']
    specialty = row['medical_specialty']
    # Prepare messages for OpenAI
    messages = [
        {"role": "system", "content": "You are an assistant in medical healthcare. Your task is to extract the age and the recommended treatment from the transcription."},
        {"role": "user", "content": transcription}
    ]
    # Extract age and recommended treatment
    try:
        age, recommended_treatment = extract_age_and_recommended_treatment(messages, client, function_definitions)
    except Exception as e:
        age, recommended_treatment = None, None
    # Get ICD-10 codes
    try:
        icd10_code = get_icd10_codes_for_treatment(recommended_treatment, client) if recommended_treatment else None
    except Exception as e:
        icd10_code = None
    structured_data.append({
        'age': age,
        'medical specialty': specialty,
        'recommended treatment': recommended_treatment,
        'ICD code': icd10_code
    })

df_structured = pd.DataFrame(structured_data)
df_structured

,age,medical specialty,recommended treatment,ICD code
0,23,Allergy / Immunology,"Try Zyrtec instead of Allegra, consider lorata...","J30.9, J45.909, J02.9, R05"
1,41,Orthopedic,operative fixation for Achilles tendon rupture,"S86.012A, S86.012D"
2,30,Bariatrics,Laparoscopic antecolic antegastric Roux-en-Y g...,"Z98.84, E66.01, E66.9"
3,50,Cardiovascular / Pulmonary,"Tracheostomy, removal of foreign body, dilatio...","J95.851, T17.9XXA, J04.9, J95.820, Z93.0"
4,66,Urology,"self-catheterization, follow up in 6 weeks, 6 ...","N31.9, N40.0, Z51.89"


In [ ]:
df.shape

(5, 2)